In [1]:
from functions import *

In [ ]:
a_vec = [763, 679, 397, 61, 697, 373, 289, 257, 625, 41, 193, 449]
b_vec = [435, 69, 330, 18, 612, 246, 496, 640, 200, 524, 672, 672]

In [1]:
def gen_cycles(max_len):
        def is_valid(r_seq, c_seq):
            length = len(r_seq)
            for i in range(length):
                if r_seq[i] == r_seq[(i+1)%length]:
                    return False
                if c_seq[i] == c_seq[(i+1)%length]:
                    return False
            return True
        def gen_utcbc():
            utcbcs = set()
            r_seq = [0, 1, 2, 1]
            for c0 in range(L):
                c2 = (c0 + 1) % l_h + (l_h if c0 >= l_h else 0)
                for c1 in range(L):
                    c3 = (c1 + 1) % l_h + (l_h if c1 >= l_h else 0)
                    c_seq = [c0, c1, c2, c3]
                    if is_valid(r_seq, c_seq):
                        positions = get_positions(r_seq, c_seq)
                        utcbcs.add(tuple(canonicalize(positions)))
            return utcbcs

        def get_positions(r_seq, c_seq):
            length = len(r_seq)
            pos = []
            for i in range(length):
                pos.append((r_seq[i], c_seq[i]))
                pos.append((r_seq[i], c_seq[(i+1)%length]))
            return pos

        def canonicalize(positions):
            symmetries = []
            curr = list(positions)
            curr_reverse = curr[::-1]
            for _ in range(len(curr)):
                curr = curr[1:] + curr[:1]
                symmetries.append(tuple(curr))
                curr_reverse = curr_reverse[1:] + curr_reverse[:1]
                symmetries.append(tuple(curr_reverse))
            return min(symmetries)

        cycles = set()
        for i in range(2, max_len//2+1):
            for r_seq in itertools.product(list(range(J)), repeat=i):
                for c_seq in itertools.product(list(range(L)), repeat=i):
                    if is_valid(r_seq, c_seq):
                        positions = get_positions(r_seq, c_seq)
                        cycles.add(tuple(canonicalize(positions)))
        cycles = cycles - gen_utcbc()
        return list(cycles)

In [ ]:

def composite_affine(left, right):
    a_new = (left[0] * right[0]) % P
    b_new = (left[0] * right[1] + left[1]) % P
    return [a_new, b_new]

def func_inv(input):
        a,b = input
        try:
            a_inv = pow(a, -1, P)
        except ValueError:
            raise ValueError("Inverse does not exist")
        b_new = (-1 * a_inv * b) % P
        return [a_inv, b_new]

In [ ]:

def generate_functions(cycles, a_vec, b_vec, h_x, h_z):
    functions = []
    for cycle in cycles:
        idx_x = [h_x[r][c] for r, c in cycle]
        idx_z = [h_z[r][c] for r, c in cycle]
        a_x = [a_vec[idx] for idx in idx_x]
        b_x = [b_vec[idx] for idx in idx_x]
        a_z = [a_vec[idx] for idx in idx_z]
        b_z = [b_vec[idx] for idx in idx_z]
        function_x = [1, 0]
        function_z = [1, 0]
        for i in range(len(cycle) // 2):
            function_x = composite_affine(function_x, [a_x[2*i], b_x[2*i]])
            function_x = composite_affine(function_x, func_inv([a_x[2*i+1], b_x[2*i+1]]))
            function_z = composite_affine(function_z, func_inv([a_z[2*i], b_z[2*i]]))
            function_z = composite_affine(function_z, [a_z[2*i+1], b_z[2*i+1]])
            
        functions.append(function_x)
        functions.append(function_z)
    return functions

In [ ]:
def get_function_x(cycles, a_vec, b_vec, h_x):
    for cycle in cycles:
        idx_x = [h_x[r][c] for r, c in cycle]
        a_x = [a_vec[idx] for idx in idx_x]
        b_x = [b_vec[idx] for idx in idx_x]
        function_x = [1, 0]
        for i in range(len(cycle) // 2):
            function_x = composite_affine(function_x, [a_x[2*i], b_x[2*i]])
            function_x = composite_affine(function_x, func_inv([a_x[2*i+1], b_x[2*i+1]]))
    return function_x

def get_function_z(cycles, a_vec, b_vec, h_z):
    for cycle in cycles:
        idx_z = [h_z[r][c] for r, c in cycle]
        a_z = [a_vec[idx] for idx in idx_z]
        b_z = [b_vec[idx] for idx in idx_z]
        function_z = [1, 0]
        for i in range(len(cycle) // 2):
            function_z = composite_affine(function_z, [a_z[2*i], b_z[2*i]])
            function_z = composite_affine(function_z, func_inv([a_z[2*i+1], b_z[2*i+1]]))
    return function_z

In [ ]:
def get_functions(cycle, a_vec, b_vec, h_x, h_z):
    f_x = get_function_x(cycle, a_vec, b_vec, h_x)
    f_z = get_function_z(cycle, a_vec, b_vec, h_z)
    return f_x, f_z

In [ ]:
def is_closed(input):
    a = input[0]
    b = input[1]
    if a == 1 and b == 0:
        return True
    d = math.gcd(a-1, P)
    if b % d == 0:
        return True
    else:
        return False

In [ ]:
def count_cycles(a_vec, b_vec, h_x, h_z):
    cycles = gen_cycles(10)
    result = {}
    for cycle in cycles:
        length = len(cycle)
        f_x, f_z = get_functions(cycle, a_vec, b_vec, h_x, h_z)
        if is_closed(f_x) or is_closed(f_z):
            if length not in result:
                result[length] = 1
            else:
                result[length] += 1
    return result